# Reviewing and Hand-Tagging Local Reports of White Terror Activity

This notebook recreates the tagging dashboard from `03_tag_local_reports.ipynb` but uses it to tag reports of local coverage of white terror organizations as compiled in `white_terror_to_review.csv`. It preserves the same workflow: review a clipping, open the Chronicling America page if needed, mark the row as `verified` or `unverifiable`, and paste the verified article text when relevant. This process is how I'm ensuring my data contains local references to white terror organizations and not general or national references.

In case you missed it, I mined sundown town and county newspapers for references to the KKK, the Red Shirts, and the White League in 04_identify_white_terror_activity.ipynb. I then subset those mined results by creating a rough context window around the given reference and checking the context window for a lexicon of 'local' words I created AND references to the corresponding county, city, and state. For more info about those processes, see 04_identify_white_terror_activity.ipynb.

Onward. The necessary libraries:

In [ ]:
import os
import html
import pandas as pd
import ipywidgets as w
from IPython.display import display

And then the dashboard, just as it was in 03_tag_local_reports.ipynb, but configured for the other .csv file:

In [ ]:
# ---------------------------
# Config
# ---------------------------
DATA_PATH = "white_terror_to_review.csv"
REQUIRED_COLS = ["search_term", "clipping", "chron_am_url"]
DISPLAY_COL = "search_term"
DISPLAY_LABEL = "Search Term"
TARGET_COL = "verified_articles"
STATUS_COL = "verification_status"  # values: pending, verified, unverifiable


# ---------------------------
# Helpers
# ---------------------------
def atomic_save(_df, path):
    tmp = path + ".tmp"
    _df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def first_unverified_pos(_df):
    idx = _df.index[_df[STATUS_COL].eq("pending")]
    return int(idx[0]) if len(idx) else 0


# ---------------------------
# Load / initialize dataframe
# ---------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Create it in the earlier notebook before running this review UI."
    )

df = pd.read_csv(DATA_PATH)

for c in REQUIRED_COLS:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.reset_index(drop=True)

if TARGET_COL not in df.columns:
    df[TARGET_COL] = pd.Series(pd.NA, index=df.index, dtype="object")
else:
    df[TARGET_COL] = df[TARGET_COL].where(pd.notna(df[TARGET_COL]), pd.NA)

if STATUS_COL not in df.columns:
    has_text = df[TARGET_COL].fillna("").astype(str).str.strip().ne("")
    df[STATUS_COL] = "pending"
    df.loc[has_text, STATUS_COL] = "verified"
else:
    valid = {"pending", "verified", "unverifiable"}
    df[STATUS_COL] = df[STATUS_COL].where(df[STATUS_COL].isin(valid), "pending")

atomic_save(df, DATA_PATH)

state = {"pos": first_unverified_pos(df)}


# ---------------------------
# Widgets
# ---------------------------
progress = w.IntProgress(min=0, max=len(df), value=0, description="Done")
row_label = w.HTML()
search_term_html = w.HTML()
clipping_html = w.HTML()
url_html = w.HTML()

decision = w.ToggleButtons(
    options=[("Verified", "verified"), ("Unverifiable", "unverifiable")],
    description="Status:",
    value="verified"
)

entry = w.Textarea(
    value="",
    placeholder="Paste verified article text here...",
    layout=w.Layout(width="100%", height="220px")
)

status = w.HTML()

btn_prev = w.Button(description="Previous")
btn_save = w.Button(description="Save")
btn_save_next = w.Button(description="Save + Next")
btn_next = w.Button(description="Next")
btn_first_unverified = w.Button(description="First Unverified")


# ---------------------------
# UI logic
# ---------------------------
def render():
    pos = state["pos"]
    row = df.iloc[pos]

    done = df[STATUS_COL].isin(["verified", "unverifiable"]).sum()
    progress.value = int(done)

    row_label.value = f"<b>Row {pos + 1} of {len(df)}</b>"

    display_text = "" if pd.isna(row[DISPLAY_COL]) else str(row[DISPLAY_COL])
    clipping_text = "" if pd.isna(row["clipping"]) else str(row["clipping"])
    url_text = "" if pd.isna(row["chron_am_url"]) else str(row["chron_am_url"])

    search_term_html.value = f"<b>{html.escape(DISPLAY_LABEL)}:</b> {html.escape(display_text)}"
    clipping_html.value = (
        "<b>Clipping:</b>"
        "<div style='white-space:pre-wrap;border:1px solid #ddd;padding:8px;margin-top:4px;'>"
        f"{html.escape(clipping_text)}</div>"
    )
    url_html.value = f'<b>chron_am_url:</b> <a href="{html.escape(url_text)}" target="_blank">{html.escape(url_text)}</a>'

    row_status = row[STATUS_COL] if row[STATUS_COL] in ("verified", "unverifiable") else "pending"
    if row_status == "unverifiable":
        decision.value = "unverifiable"
    else:
        decision.value = "verified"

    entry.value = "" if pd.isna(row[TARGET_COL]) else str(row[TARGET_COL])
    entry.disabled = (decision.value == "unverifiable")

    status.value = ""


def save_current():
    pos = state["pos"]

    if decision.value == "unverifiable":
        df.at[pos, TARGET_COL] = pd.NA
        df.at[pos, STATUS_COL] = "unverifiable"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:green;'>Saved as Unverifiable (confirmed NA).</span>"
        return True

    txt = entry.value.strip()
    if txt:
        df.at[pos, TARGET_COL] = txt
        df.at[pos, STATUS_COL] = "verified"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:green;'>Saved as Verified.</span>"
        return True
    else:
        df.at[pos, TARGET_COL] = pd.NA
        df.at[pos, STATUS_COL] = "pending"
        atomic_save(df, DATA_PATH)
        status.value = "<span style='color:#b26a00;'>Saved as Pending (blank verified text).</span>"
        return True


def on_decision_change(change):
    if change["name"] == "value":
        entry.disabled = (change["new"] == "unverifiable")


def on_prev(_):
    state["pos"] = max(0, state["pos"] - 1)
    render()


def on_next(_):
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()


def on_save(_):
    save_current()
    render()


def on_save_next(_):
    save_current()
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()


def on_first_unverified(_):
    state["pos"] = first_unverified_pos(df)
    render()


decision.observe(on_decision_change)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_save.on_click(on_save)
btn_save_next.on_click(on_save_next)
btn_first_unverified.on_click(on_first_unverified)

ui = w.VBox([
    progress,
    row_label,
    search_term_html,
    clipping_html,
    url_html,
    decision,
    w.HTML("<b>verified_articles</b>"),
    entry,
    w.HBox([btn_prev, btn_save, btn_save_next, btn_next, btn_first_unverified]),
    status
])

render()
display(ui)


## Notes on What I'm Finding

So, I've decided to post this notebook before I'm actually done tagging the results. As it currently stands, I've tagged 156 rows so far and found 71 verified articles about local KKK activity. As I've been tagging, I'm also taking notes about what I'm finding. These notes are general in nature. They are:

- not sure if this data will help parse local vs. regional activity. I might have to cue BERT into the local county name and cities, perhaps, because the local vs. national or regional reporting in these cases looks more-or-less the same
- lots of Reconstruction-era reports of KKK prosecutions
- lots of denial of local Klan activity while reporting activity in adjacent counties
- some odd reports about notable ppl discovered to be KKK members in surprising ways or by happenstance - these are kinds of vignettes
- The Chicago Whip has lots of coverage of KKK activity in Chicago in the 1920s. Will make a good point of emphasis about the varied scales of sundown towns
- lots of reports blaming KKK activity on Black ppl, saying it actually wasn't KKK, but a group of Black gang members. Amazing misinformation not unlike "Antifa" talk today
- the local reports most reliably proving to be actual local KKK activity tend to have the city or county name in the context window. This probably a remnant of my filtering of the data, but also, an easy way to ensure coverage is local.

Anyway, this process is working, but it's slower than my hand-keying of local lynching coverage. This is mainly because the language of local KKK activity is not very different than the language of KKK activity in other states or places. I have to read more closely to figure out if it's local. I'm also Google mapping places more often, to see if places of activity are or aren't local. This makes me wonder how viable this data will be. If used to fine-tune BERT, how will BERT be able to use the extra info of maps and locations I'm using to tag reports as local or not?

That's a problem I'll have to work through. But in the meantime, I'll keep tagging and I'll forge ahead with other parts of the whole process. Next up: creating a random stew of newspaper text to be BERT's negative examples (i.e., not local coverage of racial violence or white terror activity). And, testing BERT with my training data.

In [ ]:
df = pd.read_csv('white_terror_to_review.csv')

df['verification_status'].value_counts()